# Chronic Pain Prediction - Sensor-Only Oracle Tier

This notebook evaluates the model's intrinsic ability to predict pain scale levels using **only raw physiological signals**, with no knowledge of the pain type (headache, back pain, etc.).

### Sensor-Only Strategy:
1. **Subject Bio-typing**: Subjects are clustered into 4 physiological groups based on their resting sensor baselines (BVP, EDA, Temp).
2. **Discrete Wavelet Transform (DWT)**: Captures precise time-frequency signatures of pain in the BVP signal.
3. **Zero Categorical Context**: The `pain_type` feature is excluded from all steps to test the pure biometric signal strength.
4. **Research-Grade Stacking**: A 5-model ensemble meta-learned via Logistic Regression.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import pywt
from scipy.fft import fft
from scipy.signal import butter, filtfilt
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier, StackingClassifier, AdaBoostClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib
import warnings

warnings.filterwarnings('ignore')
sns.set(style="whitegrid")

## 1. Professional Preprocessing (Filtering & Bio-typing)

In [ ]:
df = pd.read_csv('cropped_dataset.csv')
fs = 4.0

def butter_filter(data, cutoff, fs, btype, order=4):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype=btype)
    return filtfilt(b, a, data)

print("Applying pre-processing filters...")
df['bvp_f'] = butter_filter(df['bvp'].values, np.array([0.3, 1.5]), fs, 'bandpass')
df['eda_f'] = butter_filter(df['eda'].values, 0.1, fs, 'lowpass')

print("Calculating Subject Bio-types (Clustering by sensor baselines only)... ")
subject_profiles = df.groupby('person_id')[['bvp', 'eda', 'temperature']].agg(['mean', 'std']).reset_index()
subject_profiles.columns = ['person_id', 'b_m', 'b_s', 'e_m', 'e_s', 't_m', 't_s']

kmeans = KMeans(n_clusters=4, random_state=42)
profiles_scaled = StandardScaler().fit_transform(subject_profiles.drop('person_id', axis=1))
subject_profiles['bio_type'] = kmeans.fit_predict(profiles_scaled)

df = df.merge(subject_profiles[['person_id', 'bio_type']], on='person_id')
print("Subject bio-typing successful.")

## 2. Subject-wise Normalization

In [ ]:
norm_cols = ['bvp_f', 'eda_f', 'x', 'y', 'z', 'temperature']
for col in norm_cols:
    means = df.groupby('person_id')[col].transform('mean')
    stds = df.groupby('person_id')[col].transform('std')
    df[col] = (df[col] - means) / (stds + 1e-8)

print("Normalization complete.")

## 3. Sensor-Only Feature Extraction (DWT & FFT)

In [ ]:
def extract_sensor_only_features(group):
    step = 25
    features = []
    p_id = group['person_id'].iloc[0]
    p_sc = group['pain_scale'].iloc[0]
    bio_ty = group['bio_type'].iloc[0]
    # NOTE: pain_type is ignored entirely
    
    prev_f = None
    
    for i in range(100, len(group) - 150, step):
        win_100 = group.iloc[i:i+100]
        
        # NO pain_type in features
        f = {'person_id': p_id, 'bio_type': bio_ty, 'pain_scale': p_sc}
        f['bvp_std_100'] = np.std(win_100['bvp_f'])
        f['eda_m_100'] = np.mean(win_100['eda_f'])
        
        # 1. Discrete Wavelet Transform (DWT)
        b_vals = win_100['bvp_f'].values
        cA, cD = pywt.dwt(b_vals, 'db4')
        f['bvp_wav_A'] = np.sum(cA**2) / len(cA)
        f['bvp_wav_D'] = np.sum(cD**2) / len(cD)
        
        # 2. FFT Energy
        f['bvp_spectral'] = np.sum(np.abs(fft(b_vals))**2) / 100
        
        # 3. Trends (t-1)
        if prev_f:
            f['eda_trend'] = f['eda_m_100'] - prev_f['eda_m_100']
        else:
            f['eda_trend'] = 0.0
            
        prev_f = f.copy()
        features.append(f)
        
    return pd.DataFrame(features)

print("Generating sensor-pure feature matrix...")
df_pure = df.groupby('person_id', group_keys=False).apply(extract_sensor_only_features).reset_index(drop=True)
print(f"Dataset size: {len(df_pure)} windows.")

## 4. Stacking Ensemble Implementation

In [ ]:
le_scale = LabelEncoder()
y = le_scale.fit_transform(df_pure['pain_scale'])

top_fcols = [c for c in df_pure.columns if c not in ['person_id', 'pain_scale']]
X = StandardScaler().fit_transform(df_pure[top_fcols])
groups = df_pure['person_id']

base_models = [
    ('rf', RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)),
    ('xgb', xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, random_state=42)),
    ('svc', SVC(probability=True, C=10, random_state=42)),
    ('mlp', MLPClassifier(hidden_layer_sizes=(128, 64), random_state=42)),
    ('ada', AdaBoostClassifier(n_estimators=100, random_state=42))
]

stack_model = StackingClassifier(estimators=base_models, final_estimator=LogisticRegression(), cv=5, n_jobs=-1)
print("Sensor-Only Stacking Ensemble Ready.")

## 5. Accuracy Validation (No Context Labels)

In [ ]:
gkf = GroupKFold(n_splits=5)
pure_scores = []

print("Validating sensor-intrinsic performance...")
for tr_i, val_i in gkf.split(X, y, groups=groups):
    stack_model.fit(X[tr_i], y[tr_i])
    pure_scores.append(accuracy_score(y[val_i], stack_model.predict(X[val_i])))

print(f"\nSensor-Only Accuracy: {np.mean(pure_scores):.4f} (+/- {np.std(pure_scores):.4f})")

## 6. Evaluation & Model Export

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
ti, tei = next(gss.split(X, y, groups=groups))
stack_model.fit(X[ti], y[ti])
yp = stack_model.predict(X[tei])

plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix(y[tei], yp), annot=True, fmt='d', cmap='viridis', 
            xticklabels=le_scale.classes_, yticklabels=le_scale.classes_)
plt.title('Sensor-Only Model Confusion Matrix')
plt.show()

pure_artifacts = {
    'model': stack_model, 'kmeans': kmeans, 
    'scaler': StandardScaler().fit(df_pure[top_fcols]), 
    'le_scale': le_scale, 'features': top_fcols
}
joblib.dump(pure_artifacts, 'pain_model_no_type.joblib')
print("Sensor-Only model saved successfully.")